# GDL - Midterm n.1
**Student ID: 583309** (Your student "matricola" goes here)

**Submission ID: 9** (The ID from the sheet circulated in classroom goes here)

In the first midterm assignment, you are required to implement a Gaussian Mixture Model (GMM) and the Expectation-Maximization (EM) algorithm. You are allowed to use `numpy` and other non-machine learning libraries, e.g., `pandas`, `matplotlib`.

**Assumptions.** To ease the implementation, we assume, for each Gaussian distribution in the mixture $\mathcal{N}(\mu_k, \Sigma_k)$, that the *covariance matrix is diagonal*. Furthermore, keep in mind that a good implementation of the EM algorithm provides monotonically increasing log-likelihood, *but* can get stuck in local minima; initialization strategies are fundamental (random? sample points? k-means++?).

**Hyperparameter $k$.** When using a GMM, you should ask yourself: *how many mixtures will be in the data?* You can *maximize* the Bayesian Information Criterion (BIC) to select the number of categories $k$ on your training set. Formally,
$$
\text{BIC}
=
\log P(X\mid \theta) - \frac{|\theta|}{2}\log n,
$$
where $n$ is the number of samples in the training set, $\theta=\{\pi,\mu,\sigma\}$ the parameters of the GMM and $|\theta|$ the number of parameters, i.e., the sum of the number of parameters in $\pi$, $\mu$, and $\sigma$ (*hint*: it also depends on the dimensionality of data $d$!).

**Summary.** Overall, you are required to:
1.  **Implement the GMM class**. Fill the `log_likelihood(samples)`.
2. **Implement the EM algorithm**. Fill the `fit(samples)` method to fit the parameters of a GMM.
3. **Implement the BIC score**. Fill the `bic(samples)` method to perform model selection.
4. **Run training and evaluation**. Select the best scoring model using BIC (Bayesian Information Criterion) for values of $k=\{1,\ldots,6\}$.

**Evaluation.** Your solution will be tested against an hidden test set. For your learning experience, we require you to refrain from using LLM generated code. Violations will be flagged and invalidate the midterm. **Do not alter Sections 3 and 4 of the notebook.**

### 1. Libraries

In [1]:
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)

print("Libraries imported successfully.")

Libraries imported successfully.


### 2. Gaussian Mixture Model (GMM)

Feel free to play around the implementation. **For evaluation purposes**, the class **must** expose the following attributes and methods:
- `_pi: np.ndarray`
- `_mu: np.ndarray`
- `_sigma: np.ndarray`
- `fit(samples: np.ndarray)`
- `bic(samples: np.ndarray) -> np.ndarray`
- `log_likelihood(samples: np.ndarray) -> np.ndarray`

In [ ]:
def gaussian(mu: np.ndarray, sigma: np.ndarray, x: np.ndarray) -> np.ndarray:
    diff = x - mu
    exponent = -(diff**2) / (2 * sigma**2)
    pdf = np.exp(exponent) / (np.sqrt(2 * np.pi) * sigma)

    return pdf


class GaussianMixtureModel:
    def __init__(self, n_categories: int, n_features: int):
        self.n_categories = n_categories
        self.n_features = n_features

        # every cluster equiprobable
        self._pi = np.ones(shape=n_categories) * (1 / n_categories)

        # means all at zero
        self._mu = np.zeros(shape=(self.n_categories, self.n_features))

        # variances all at one to prevent initial division by zero
        self._sigma = np.ones((self.n_categories, self.n_features))

    def log_likelihood(self, samples: np.ndarray) -> np.ndarray:
        """Computes logP(X|θ)"""

        probabilities = np.zeros(shape=(samples.shape[0]))
        for i, x in enumerate(samples):
            weighted = np.zeros(self.n_categories)

            for m in range(self.n_categories):
                pdfs = gaussian(self._mu[m], self._sigma[m], x)
                proba = np.prod(pdfs)
                weighted[m] = proba * self._pi[m]

            probabilities[i] = np.log(np.sum(weighted))

        return np.asarray(np.sum(probabilities))

    def fit(self, samples: np.ndarray):
        """Fits the GMM parameters using the EM algorithm."""
        n_samples = samples.shape[0]

        # means initialization via random sampling
        indices = np.random.choice(n_samples, self.n_categories, replace=False)
        self._mu = samples[indices]

        # variance initialized with dataset variance
        variance = np.var(samples, axis=0)
        self._sigma = np.sqrt(variance)[None, :] * np.ones(
            (self.n_categories, self.n_features)
        )

        # for convergence check
        self.log_likelihood_history = [self.log_likelihood(samples)]

        for _ in range(200):
            # E-step
            responsibilities = np.zeros(shape=(n_samples, self.n_categories))

            for i, x in enumerate(samples):
                weighted = np.zeros(shape=(self.n_categories))
                for m in range(self.n_categories):
                    # for each sample compute the probability of each gaussian
                    pdfs = gaussian(self._mu[m], self._sigma[m], x)

                    # exploit conditional independence
                    proba = np.prod(pdfs)

                    weighted[m] = proba * self._pi[m]

                responsibilities[i] = weighted / np.sum(weighted)

            # M-step
            Nm = np.sum(responsibilities, axis=0)
            for m in range(self.n_categories):
                # update pi
                self._pi[m] = Nm[m] / n_samples

                # update mu
                mu_num = np.zeros(self.n_features)
                for i in range(n_samples):
                    mu_num += responsibilities[i, m] * samples[i]
                self._mu[m] = mu_num / Nm[m]

                # update sigma
                sigma_num = np.zeros(self.n_features)
                for i in range(n_samples):
                    diff = samples[i] - self._mu[m]
                    sigma_num += responsibilities[i, m] * (diff**2)
                self._sigma[m] = np.sqrt(sigma_num / Nm[m])

            curr_log_likelihood = self.log_likelihood(samples)

            # convergence check
            if abs(curr_log_likelihood - self.log_likelihood_history[-1]) < 1e-4:
                return

            self.log_likelihood_history.append(curr_log_likelihood)

    def bic(self, samples: np.ndarray) -> float:
        """Computes the BIC score."""
        n, d = samples.shape
        k = self.n_categories

        log_likelihood = self.log_likelihood(samples)

        # number of parameters
        p = 2 * k * d + (k - 1)

        return log_likelihood - 0.5 * p * np.log(n)

### 3. Training

Trains the model for increasing number of categories and selects the best scoring one in terms of BIC score.

In [3]:
# load training set
train_df = pd.read_csv('train.csv')
print(f"train.csv loaded successfully. n={train_df.shape[0]}, d={train_df.shape[1]}")

# model selection using bayesian information score
seed_everything(9951)
candidate_categories = range(1, 7)
max_bic, best_k, best_gmm = -np.inf, None, None
for k in candidate_categories:
    gmm = GaussianMixtureModel(n_categories=k, n_features=train_df.shape[1])
    gmm.fit(train_df.values)
    ll = gmm.log_likelihood(train_df.values)
    bic_score = gmm.bic(train_df.values)
    print(f"k={k}\tBIC={bic_score:.4f}\tlogP(X|θ)={ll:.4f}")
    if bic_score > max_bic:
        max_bic = bic_score
        best_k = k
        best_gmm = gmm

# print training info
print()
print("Best model")
print(f"k:\t\t{best_k}")
print(f"BIC:\t\t{max_bic:.4f}")
print(f"logP(X|θ):\t{best_gmm.log_likelihood(train_df.values):.4f}")

train.csv loaded successfully. n=800, d=5
k=1	BIC=-8320.1771	logP(X|θ)=-8286.7541
k=2	BIC=-8052.0417	logP(X|θ)=-7981.8532
k=3	BIC=-7919.3844	logP(X|θ)=-7812.4306
k=4	BIC=-7869.8688	logP(X|θ)=-7726.1497
k=5	BIC=-7898.7322	logP(X|θ)=-7718.2477
k=6	BIC=-7926.3627	logP(X|θ)=-7709.1128

Best model
k:		4
BIC:		-7869.8688
logP(X|θ):	-7726.1497


### 4. Evaluation

To check if you did not break the API, you can use the training file to run the tests.

This is not meaningful for the evaluation of your model, as the true test set is hidden.

In [4]:
# If test.csv does not exist copy train into test
!test -f test.csv || cp train.csv test.csv

In [5]:
# load hidden test set ☠️
test_df = pd.read_csv('test.csv')
print(f"test.csv loaded successfully. n={test_df.shape[0]}, d={test_df.shape[1]}")

# print log-likelihood
test_log_likelihood = best_gmm.log_likelihood(test_df.values)
print(f"Log-likelihood of test data: {test_log_likelihood:.4f}")

# print parameters
print(f"k: {best_gmm.n_categories}")
print()
for cat_id in range(best_gmm.n_categories):
  print(f"π[{cat_id}]:", best_gmm._pi[cat_id])
  print(f"μ[{cat_id}]:", best_gmm._mu[cat_id])
  print(f"σ[{cat_id}]:", best_gmm._sigma[cat_id])
  print()

test.csv loaded successfully. n=800, d=5
Log-likelihood of test data: -7726.1497
k: 4

π[0]: 0.32349174429453925
μ[0]: [-2.76487659 -0.57500293 -0.45659567  2.05101651 -3.81371502]
σ[0]: [1.25088359 1.77692787 1.02567142 1.48515925 1.2496463 ]

π[1]: 0.12069528452092738
μ[1]: [ 1.8792873   2.39719924 -0.19697045  0.26456917 -0.80136179]
σ[1]: [0.70177675 1.32863968 1.17937328 1.24612778 1.80192147]

π[2]: 0.12061816523572493
μ[2]: [-2.9487705  -1.75826876  0.81574653  2.46376901 -0.09864943]
σ[2]: [1.52698414 1.60566011 0.8731689  1.539611   0.8126766 ]

π[3]: 0.4351948059488089
μ[3]: [-2.01544035  0.56098144 -2.33851178 -0.50666027 -0.10951374]
σ[3]: [1.60110057 1.49969274 1.10968356 1.28403481 1.66993157]

